# pobblebonk

In [ ]:
#| hide
from pobblebonk.core import *
from pobblebonk.pob import *
from pobblebonk.run import *

> the clock and the notebook for an agent

pobblebonk remembers what you asked for and when it is due. It does not do the work. One SQLite
file holds four things: a durable queue, your schedules, the lists you add to over a week, and the
notes waiting for you. An agent does the rest.

It exists because two jobs get mixed up. Running a reminder is one job. Telling you about it is
another, and the second is harder. In the Agent Client Protocol the editor asks first and the
agent answers. An agent cannot start a turn. So nothing outside your editor can put a 5pm roundup
into the conversation you have open.

pobblebonk runs a fire on its own instead. It keeps the answer and leaves a note. Each place you
work picks that note up on its next turn.

## Install

```sh
uv add pobblebonk        # add pobblebonk[acp] for the ACP runner
```

## The shape of it

```
you                    pobblebonk                  an agent
---                    ----------                  --------
add to a list  ------>  pob_items
write a schedule ---->  pob_schedules
                        |
cron, once a minute --> tick ---> jobs ----------> one prompt over ACP
                                   |                    |
                        pob_notes <-- the answer -------+
                        |
type anything <--------- drain
```

## A week of shopping, and Wednesday night

Send things to a list whenever you think of them. Write the schedule once.

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

wed = datetime(2026, 8, 26, 20, 1, tzinfo=ZoneInfo('Asia/Kolkata')).timestamp()

pob = Pob().attach(FakeRunner(['added 6 items, no oat milk']))
pob.lists.add('shopping', 'yoghurt, the greek one', meta=dict(photo='img_2201.heic'))
pob.lists.add('shopping', 'oat milk')

pob.scheds.add('cart', 'Fill my cart from the shopping list.',
               cron='0 20 * * 3', tz='Asia/Kolkata', needs='shopping', policy='write')
pob

Pob(1 schedules, 0 ready, 0 dead, 0 notes)

`tick` is what a cron entry calls. It fires what is due, runs it, and leaves the answer.

In [ ]:
out = pob.tick(at=wed)
out.ran[0].status, out.ran[0].result

('ok',
 {'text': 'added 6 items, no oat milk',
  'items': 2,
  'session': 'fake-1',
  'refused': []})

Next time you type anything, in any editor, the note is there. Each reader sees it once.

In [ ]:
pob.drain('leela')[0].body

'added 6 items, no oat milk'

## A roundup that remembers last week

Keep the session the agent hands back and tomorrow's fire continues the same conversation. That is
what makes a trend cheap: the agent still has last week.

In [ ]:
pob.scheds.add('git roundup', 'Roundup my commits today and say how that compares with last week.',
               cron='0 17 * * *', tz='Asia/Kolkata', start=wed)
pob.tick(at=wed)
pob.scheds.get('git roundup')['session']    # kept after the first fire, and reused by the next

'fake-1'

## A digest that needs to know your interests

Nothing can ask you a question at 7am. So a schedule that needs a list it has not got does not
run, and does not guess. It says what is missing, once, and starts again on its own when the list
fills.

In [ ]:
pob2 = Pob().attach(FakeRunner())
pob2.scheds.add('news', 'The news that matters, based on my interests.',
                every='1d', needs='interests', start=0)

pob2.tick(at=1).blocked, pob2.inbox.notes(kind='blocked')[0].title

(['news'], 'news needs your interests list')

In [ ]:
pob2.lists.add('interests', 'sanskrit grammar')
out = pob2.tick(at=90_000)
out.freed, len(out.fired)

(['news'], 1)

## What is in it

| module | what it holds |
|----|----|
| [`core`](00_core.ipynb) | the vault, and the small `Store` the four tables are built on |
| [`queue`](01_queue.ipynb) | at-least-once work: retries, leases, a dead letter, and what each run answered |
| [`clock`](02_clock.ipynb) | durations, cron in a named zone, and the catch-up policy |
| [`lists`](03_lists.ipynb) | the piles you add to, and who is allowed to empty them |
| [`inbox`](04_inbox.ipynb) | notes, with a cursor per reader |
| [`schedule`](05_schedule.ipynb) | the standing instruction, and the tick that fires it |
| [`run`](06_run.ipynb) | handing one fire to an ACP agent, and keeping what it said |
| [`pob`](07_pob.ipynb) | all four on one vault |
| [`cli`](08_cli.ipynb) | `pob tick`, and the commands for looking |

## What is deliberately not in it

No daemon and no broker. Whoever has the file does the work. Something that already exists calls
`pob tick`: a cron entry, a launchd plist, a systemd timer, a button.

No driving. pobblebonk owns the clock, not the work. A runner is anything callable that takes a
prompt and answers with text. Nothing here imports the agent it starts.

No lock table. The data settles two ticks racing. One `BEGIN IMMEDIATE` covers reading a schedule
and advancing it, and each fire is queued under `<schedule>:<fire time>`, which a unique index
refuses twice. Four processes ticking at once queue one job.

No push. `tick` finds what is due. Millisecond wakeups on commit are
[honker](https://github.com/russellromney/honker). A weekly reminder does not need them.

## Sharing a vault

Every class takes a database rather than opening one. Hand it a vault that
[litesearch](https://github.com/Karthik777/litesearch) already has open. The queue then shares that
connection, its lock and its transaction. A document write and the job it caused commit together,
and roll back together.

```python
from litesearch import Index
ix = Index('vault.db')
pob = Pob(db=ix.db)          # one file, one transaction
```

## The tick

Once a minute is plenty, and a tick with nothing due is one indexed read.

```sh
* * * * * pob tick --db ~/.pobblebonk/vault.db
```

A laptop shut for nine days comes back with nine missed fires. Each schedule's `catchup` decides
what happens to them.

- `once` runs the latest and forgets the rest. This is the default.
- `all` runs every one.
- `skip` runs none and only moves the clock on.

The default is `once` for a reason. Reopening a laptop after a holiday should not empty the whole
week into your cart.

## Develop

```sh
uv sync --all-extras --group dev
uv run nbdev-export      # notebooks -> pobblebonk/*.py
uv run nbdev-test        # execute every notebook
uv run pytest            # the plain-python suite
uv run nbdev-clean       # before committing
```

`nbs/*.ipynb` is the source. Every module in `pobblebonk/` is generated and says so at the top.